# 05 - Optuna TPE: Hiperparametros de entrenamiento

Busqueda secuencial por sujeto, con el mismo `pipeline_id` del 02. Ejecutar solo despues de V01-V14 PASS y de decidir batch y epocas con el 04. Los candidatos del piloto no habilitan esta busqueda.

El bucle muestra sujetos, trials, folds y epocas. Siempre puntua la ultima epoca: sin early stopping, pruning ni seleccion del mejor pico. Solo desarrollo; no se abren etiquetas de test.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/SaulRodas/BayesianOptimizationNN_ECOG.git'
BRANCH = 'main'
PROJECT = Path('/content/tt-ecog')
PREVIOUS = None
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPOSITORY, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('El checkout existente pertenece a otro remoto; revisa /content/tt-ecog.')
    if subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('El clon de Colab tiene cambios locales; no se actualiza automáticamente.')
    PREVIOUS = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if 'tt_ecog' in sys.modules and PREVIOUS is not None and PREVIOUS != COMMIT:
    raise RuntimeError('El código cambió después de importarlo: reinicia el runtime y vuelve a ejecutar.')
print('Commit en Colab:', COMMIT)
print('Debe coincidir con `git rev-parse HEAD` local (haz push antes de ejecutar).')

In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'scripts/check_environment.py'), '--install-missing'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT) + '[search]'], check=True)
# El .pth del modo editable solo se procesa al iniciar el intérprete; el kernel activo no lo ve.
SOURCE = PROJECT / 'src'
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import tt_ecog
import torch
if not Path(tt_ecog.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError(f'tt_ecog se importa desde {tt_ecog.__file__}, fuera del checkout; reinicia el runtime.')
print('tt_ecog', tt_ecog.__version__, '| torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')
CONFIG = PROJECT / 'configs/experiment_config.yaml'


## Configuracion, datos aprobados y GPU

In [ ]:
import gc
from time import perf_counter
import pandas as pd
import torch
from tt_ecog.config import load_config
from tt_ecog.datasets import load_fold
from tt_ecog.io import approved_pipeline, save_training_checkpoint, load_training_checkpoint
from tt_ecog.model import build_model
from tt_ecog.losses import make_loss
from tt_ecog.training import (seed_run, make_loaders, make_optimizer, train_epoch, evaluate,
    device_info, reset_peak_memory, peak_memory, synchronize)
from tt_ecog.optimization import (search_budget, search_session, next_trial, suggest_parameters,
    trial_training_run, record_fold, complete_trial, fail_trial)
from tt_ecog.reporting import (record_training_epoch, finish_training_run, report_search,
    record_training_resume, restore_epoch_metrics, utc_now)
config = load_config(CONFIG)
STAGE = 'stage1'

In [ ]:
batch_size, epochs = search_budget(config)
pipeline_dir = approved_pipeline(config, DRIVE_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError('Activa la GPU de Colab antes de ejecutar la busqueda.')
device = torch.device('cuda')
print(device_info(device))
print('Batch comun:', batch_size, '| epocas fijas:', epochs)

## Presupuesto y recuperacion

20 trials por sujeto; cada uno valida B2 y B4. TPE propone learning rate y weight decay en escala logaritmica; la arquitectura de referencia permanece fija.

SQLite trabaja en disco local; `study.zip` en Drive conserva juntos la base y el estado del sampler. Usar un solo runtime por estudio. Al reabrir se recuperan el trial en curso, sus folds terminados y la ultima epoca completa del fold pendiente. No se agregan trials por volver a ejecutar el notebook.

Una desconexion o interrupcion permite continuar; un fallo numerico, OOM o error de datos queda como FAIL y detiene la busqueda. No se reintenta ni se reemplaza automaticamente. No cambiar epochs, batch o configuracion dentro de un estudio. Cargar solo los archivos de estudio propios: el sampler serializado requiere la misma version de Optuna.

In [ ]:
summaries = []
for subject in config.data['project']['subjects']:
    with search_session(config, DRIVE_ROOT, subject, STAGE) as session:
        study = session.study
        print(subject, STAGE, '|', session.contract['pipeline_id'])
        try:
            while True:
                trial = next_trial(session)
                if trial is None:
                    break
                output, history = None, []
                try:
                    arch, lr, weight_decay = suggest_parameters(session, trial)
                    session.save()
                    print('Trial', trial.number, arch.key, '| lr:', lr, '| wd:', weight_decay)
                    for val_block in session.contract['policy']['validation_blocks']:
                        if val_block in trial.user_attrs.get('folds', {}):
                            print(val_block, 'ya terminado; se conserva su resultado.')
                            continue
                        output, history = None, []
                        fold = load_fold(config, pipeline_dir / 'folds', subject, val_block, verify=False)
                        output, parameters = trial_training_run(session, trial, fold, arch, lr, weight_decay, device)
                        seed_run(parameters['seed'])
                        train_loader, val_loader = make_loaders(fold, batch_size, parameters['seed'])
                        model = build_model(config, fold.blocks[val_block].x.shape[0], arch).to(device)
                        optimizer = make_optimizer(model, lr, weight_decay)
                        loss_fn = make_loss(config)
                        checkpoint = output / 'checkpoint/last.pt'
                        completed = 0
                        if checkpoint.exists():
                            state = load_training_checkpoint(checkpoint, model, optimizer, train_loader, fold.pipeline_id)
                            if state['parameters'] != parameters:
                                raise ValueError('El checkpoint no corresponde a los parametros de este trial.')
                            completed, history = state['epoch'], state['history']
                            del state
                            record_training_resume(output, device, completed)
                        restore_epoch_metrics(output, history)

                        for epoch in range(completed + 1, epochs + 1):
                            reset_peak_memory(device)
                            synchronize(device)
                            started = perf_counter()
                            train_loss = train_epoch(model, optimizer, loss_fn, train_loader, device)
                            metrics = evaluate(model, loss_fn, fold, val_loader, device,
                                               config.data['metrics']['degenerate_std_relative'])
                            synchronize(device)
                            row = record_training_epoch(output, parameters, epoch, train_loss, metrics,
                                                        perf_counter() - started, peak_memory(device))
                            history.append(row)
                            save_training_checkpoint(checkpoint, model, optimizer, epoch, parameters,
                                                     train_loader, history=history)
                            print(f"{subject} trial {trial.number} {val_block} | epoca {epoch}/{epochs} | "
                                  f"Pearson {row['val_pearson_full']:.4f}")

                        finish_training_run(output, parameters, 'COMPLETE', epochs, history[-1]['val_pearson_full'])
                        record_fold(session, trial, val_block, history, parameters, device_info(device))
                        session.save()
                        del model, optimizer, train_loader, val_loader, fold
                        output, history = None, []
                        gc.collect()
                        if device.type == 'cuda':
                            torch.cuda.empty_cache()
                    score = complete_trial(session, trial)
                    print('Objetivo: media de los folds =', score)
                except KeyboardInterrupt:
                    trial.set_user_attr('interrupted_at_utc', utc_now())
                    session.save()
                    if output is not None:
                        finish_training_run(output, parameters, 'INTERRUPTED', len(history),
                            history[-1]['val_pearson_full'] if history else None,
                            details='Recuperar desde el ultimo checkpoint completo.')
                    raise
                except Exception as exc:
                    if output is not None:
                        finish_training_run(output, parameters, 'FAILED', len(history),
                            history[-1]['val_pearson_full'] if history else None, details=str(exc))
                    fail_trial(session, trial, exc)
                    raise
        finally:
            summary = report_search(session)
            summaries.append(summary)
            print('Estado:', summary['status'], '| resultados:', session.archive.folder)


## Resultados

Los resultados por fold y por dedo, curvas por epoca, GPU y semillas quedan en los reportes y checkpoints. El resumen selecciona parametros solo cuando todos los trials presupuestados terminaron. Este notebook no ejecuta evaluacion final.

In [ ]:
display(pd.DataFrame([{'subject': s['subject'], 'stage': s['stage'], 'status': s['status'],
                       'selection': s['selection']} for s in summaries]))